# [CS303]Practice 5 - Adversarial Search
For this practice, complete the required search algorithms below and make sure that the provided tests run correctly.

#### **Required Tasks**
1. **Minimax Search:** Complete the `TODO` sections in the Minimax implementation.
2. **Alpha-Beta Search:** Complete the `TODO` sections in the Alpha-Beta pruning implementation. The algorithm must return the same optimal decision as Minimax.

#### **Optional Task**
3. **Cutoff Search:** Optionally complete the depth-limited Alpha-Beta search with a cutoff test and an evaluation function.
                                                                                                      
#### **Practice Deadline and Checking Policy**

The score for this practice is determined by **when you have your work checked during your lab session**. Please follow the schedule of your own lab group.

| Lab Group     | Full Credit (100%)           | Late Check (50%)        | After the 50% Deadline |
| ------------- | ---------------------------- | ----------------------- | ---------------------- |
| **Wednesday** | **Oct. 14** or **Oct. 21**   | **Oct. 28**             | 0%                     |
| **Thursday**  | **Oct. 15** or **Oct. 22**   | **Oct. 29**             | 0%                     |
| **Friday**    | **Oct. 16** or **Oct. 23**   | **Oct. 30**             | 0%                     |

You have **two lab sessions to complete and have your work checked for full credit (100%)**. If you miss both sessions, you can still have your work checked at the designated late-check session, but you will receive only **50% of the points**. Any check after the 50% deadline will receive **0 points**.

**Important:** The score is determined by the **date of your in-person check**.

#### **What will be checked**

- You understand every line of your code.
- Your program runs without errors.


In [21]:
from game import TicTacToe
import random
import math
import time
infinity = math.inf

# 1. Getting Started with the Game

In [22]:
# initializing a TicTacToe game
game = TicTacToe(height=3, width=3, k=3)
board = game.initial
# get the possible actions
actions = game.actions(board)
# to apply a move (the board automatically figures out which player)
board_after_move = game.result(board, list(actions)[0])

print(board)
print(actions)
print(board_after_move)

. . .
. . .
. . .

{(0, 1), (1, 2), (2, 1), (0, 0), (1, 1), (2, 0), (0, 2), (2, 2), (1, 0)}
. . .
X . .
. . .



In [23]:
def play_game_step(game, state, strategies: dict, verbose=False):
    start = time.perf_counter()
    player = state.to_move
    move = strategies[player](game, state)
    state = game.result(state, move)
    time_elapsed = time.perf_counter() - start
    if verbose: 
        print('Player', player, 'move:', move, f'time: {time_elapsed:.4f}s', )
        print(state)
    return state

def play_game(game, strategies: dict, verbose=False):
    """Play a turn-taking game. `strategies` is a {player_name: function} dict,
    where function(state, game) is used to get the player's move."""
    state = game.initial
    while not game.is_terminal(state):
        state = play_game_step(game, state, strategies, verbose)
    return state

# setup a random strategy for testing
def random_player(game, state): return random.choice(list(game.actions(state)))

def search_player(search_algorithm):
    """A game player who uses the specified search algorithm"""
    return lambda game, state: search_algorithm(game, state)[1] # we expect our search algorithm to return (v, move)

In [24]:
# initialize a game
game = TicTacToe()
state = game.initial
strategies = dict(X=random_player, O=random_player)

In [25]:
# option 1: 
# play_game(game, strategies, True)

# option 2: run it step by step by executing this cell multiple times until the game end
if not game.is_terminal(state):
    state = play_game_step(game, state, strategies, True)
else:
    print(f"Game result: {game.utility(state, 'X')}")
    print(state)

Player X move: (0, 1) time: 0.0000s
. . .
X . .
. . .



# 2. MINIMAX and Alpha-Beta 

### MINIMAX-DECISION and EXPECTIMINIMAX

__function__ MINIMAX-DECISION(_state_) __returns__ _an action_  
&emsp;__return__ arg max<sub> _a_ &Element; ACTIONS(_s_)</sub> MIN\-VALUE(RESULT(_state_, _a_))  

---
__function__ MAX\-VALUE(_state_) __returns__ _a utility value_  
&emsp;__if__ TERMINAL\-TEST(_state_) __then return__ UTILITY(_state_)  
&emsp;_v_ &larr; &minus;&infin;  
&emsp;__for each__ _a_ __in__ ACTIONS(_state_) __do__  
&emsp;&emsp;&emsp;_v_ &larr; MAX(_v_, MIN\-VALUE(RESULT(_state_, _a_)))  
&emsp;__return__ _v_  

---
__function__ MIN\-VALUE(_state_) __returns__ _a utility value_  
&emsp;__if__ TERMINAL\-TEST(_state_) __then return__ UTILITY(_state_)  
&emsp;_v_ &larr; &infin;  
&emsp;__for each__ _a_ __in__ ACTIONS(_state_) __do__  
&emsp;&emsp;&emsp;_v_ &larr; MIN(_v_, MAX\-VALUE(RESULT(_state_, _a_)))  
&emsp;__return__ _v_  

---
__function__ EXPECTIMINIMAX(_s_) =     
&emsp;UTILITY(_s_) __if__ TERMINAL\-TEST(_s_)  
&emsp;max<sub>_a_</sub> EXPECTIMINIMAX(RESULT(_s, a_)) __if__ PLAYER(_s_)= MAX  
&emsp;min<sub>_a_</sub> EXPECTIMINIMAX(RESULT(_s, a_)) __if__ PLAYER(_s_)= MIN  
&emsp;∑<sub>_r_</sub> P(_r_) EXPECTIMINIMAX(RESULT(_s, r_)) __if__ PLAYER(_s_)= CHANCE  


In [26]:
def minimax_search(game, state):
    """Search game tree to determine best move; return (value, move) pair."""

    player = state.to_move

    def max_value(state):
        # utility has three possible values:
        # +1: player wins the game
        # -1: player loses the game
        # 0: no winner -> a draw
        if game.is_terminal(state):
            return game.utility(state, player), None

        # set initial value to negative infinity,
        # so that any value returned by min_value would be greater than it
        # this is a typical trick to find max
        v, move = -infinity, None

        # loop through all possible actions, and find the one that maximizes the value
        for a in game.actions(state):

            # for each action, we get the value of the resulting state by calling min_value,
            # as it's the opponent's turn after we make this move
            # for our opponent, he / she will try to minimize the value
            v2, _ = min_value(game.result(state, a))

            # TODO: decide *v* and *move*
            # this value is greater than current best,
            # we update the best value and record the move that led to this value
            if v2 > v:
                v, move = v2, a

        # after the whole loop,
        # best value and the corresponding move are recorded in v & move
        # so for future max_value calls, we can return for this state
        return v, move

    def min_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), None

        # this time we set initial value to positive infinity,
        # cuz we are going to find the minimum one,
        # and any value we met is granted to be smaller than positive infinity
        v, move = +infinity, None

        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a))

            # TODO: decide *v* and *move*

            # update best if the value is samller than current best,
            # and record the move that led to this value
            if v2 < v:
                v, move = v2, a

        return v, move

    return max_value(state)

# test against random_player
%time play_game(TicTacToe(), dict(X=search_player(minimax_search), O=random_player), True)

Player X move: (0, 1) time: 2.1862s
. . .
X . .
. . .

Player O move: (1, 2) time: 0.0000s
. . .
X . .
. O .

Player X move: (1, 1) time: 0.0283s
. . .
X X .
. O .

Player O move: (2, 2) time: 0.0000s
. . .
X X .
. O O

Player X move: (2, 1) time: 0.0004s
. . .
X X X
. O O

CPU times: user 2.13 s, sys: 20.1 ms, total: 2.15 s
Wall time: 2.22 s


. . .
X X X
. O O

In [27]:
# if you have implemented it right, it will always be a draw for h=w=k=3
%time play_game(TicTacToe(), dict(X=search_player(minimax_search), O=search_player(minimax_search)), True)

Player X move: (0, 1) time: 2.1983s
. . .
X . .
. . .

Player O move: (2, 1) time: 0.2899s
. . .
X . O
. . .

Player X move: (1, 2) time: 0.0382s
. . .
X . O
. X .

Player O move: (0, 0) time: 0.0057s
O . .
X . O
. X .

Player X move: (1, 1) time: 0.0011s
O . .
X X O
. X .

Player O move: (1, 0) time: 0.0002s
O O .
X X O
. X .

Player X move: (2, 0) time: 0.0001s
O O X
X X O
. X .

Player O move: (0, 2) time: 0.0000s
O O X
X X O
O X .

Player X move: (2, 2) time: 0.0000s
O O X
X X O
O X X

CPU times: user 2.47 s, sys: 22.8 ms, total: 2.49 s
Wall time: 2.53 s


O O X
X X O
O X X

### ALPHA-BETA-SEARCH

__function__ ALPHA-BETA-SEARCH(_state_) __returns__ an action  
&emsp;_v_ &larr; MAX\-VALUE(_state_, &minus;&infin;, &plus;&infin;)  
&emsp;__return__ the _action_ in ACTIONS(_state_) with value _v_  

---
__function__ MAX\-VALUE(_state_, _&alpha;_, _&beta;_) __returns__ _a utility value_  
&emsp;__if__ TERMINAL\-TEST(_state_) __then return__ UTILITY(_state_)  
&emsp;_v_ &larr; &minus;&infin;  
&emsp;__for each__ _a_ __in__ ACTIONS(_state_) __do__  
&emsp;&emsp;&emsp;_v_ &larr; MAX(_v_, MIN\-VALUE(RESULT(_state_, _a_), _&alpha;_, _&beta;_))  
&emsp;&emsp;&emsp;__if__ _v_ &ge; _&beta;_ __then return__ _v_  
&emsp;&emsp;&emsp;_&alpha;_ &larr; MAX(_&alpha;_, _v_)  
&emsp;__return__ _v_  

---
__function__ MIN\-VALUE(_state_, _&alpha;_, _&beta;_) __returns__ _a utility value_  
&emsp;__if__ TERMINAL\-TEST(_state_) __then return__ UTILITY(_state_)  
&emsp;_v_ &larr; &plus;&infin;  
&emsp;__for each__ _a_ __in__ ACTIONS(_state_) __do__  
&emsp;&emsp;&emsp;_v_ &larr; MIN(_v_, MAX\-VALUE(RESULT(_state_, _a_), _&alpha;_, _&beta;_))  
&emsp;&emsp;&emsp;__if__ _v_ &le; _&alpha;_ __then return__ _v_  
&emsp;&emsp;&emsp;_&beta;_ &larr; MIN(_&beta;_, _v_)  
&emsp;__return__ _v_  


In [28]:
def alphabeta_search(game, state):
    """Search game to determine best action; use alpha-beta pruning.
    As in [Figure 5.7], this version searches all the way to the leaves."""

    # we need to know which player is making the move at this state,
    # so that we can evaluate the utility of the terminal states,
    # from the perspective of this player.
    player = state.to_move

    # first define:
    # 1. alpha: the best value that the maximizer currently can guarantee
    # that is, the maximizer can at least get this value, no matter what the minimizer does
    # 2. beta: the best value that the minimizer currently can guarantee
    # that is, the minimizer can let the maximizer get at most this value,
    # as if later we find a value that is greater than beta,
    # the minimizer will never let us get this value,
    # so we can prune the rest of the branches
    def max_value(state, alpha, beta):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):

            # after implementing the current move a, 
            # we dive into the our opponent's turn, who is the minimizer,
            # so we call min_value to get the value of the resulting state.
            v2, _ = min_value(game.result(state, a), alpha, beta)

            # TODO: update alpha, beta pruning, decide *v* and *move*
            
            # update best:
            if v2 > v:
                v, move = v2, a

            # if current best is greater or equal to beta,
            # we can prune the rest of the branches, as our opponent will never let us get this value
            if v >= beta:
                return v, move

            # update alpha, the best value that the maximizer currently can guarantee
            alpha = max(alpha, v)
            
        return v, move

    def min_value(state, alpha, beta):
        # TODO: implement function min_value

        if game.is_terminal(state):
            return game.utility(state, player), None

        v = infinity
        move = None

        for a in game.actions(state):
            v2, _ = max_value(
                game.result(state, a),
                alpha,
                beta
            )

            if v2 < v:
                v, move = v2, a

            if v <= alpha:
                return v, move
            beta = min(beta, v)
            
        return v, move

    return max_value(state, -infinity, +infinity)

# test against random_player
%time play_game(TicTacToe(), dict(X=search_player(alphabeta_search), O=random_player), True)

Player X move: (0, 1) time: 0.0971s
. . .
X . .
. . .

Player O move: (0, 2) time: 0.0000s
. . .
X . .
O . .

Player X move: (1, 2) time: 0.0048s
. . .
X . .
O X .

Player O move: (1, 0) time: 0.0000s
. O .
X . .
O X .

Player X move: (2, 1) time: 0.0004s
. O .
X . X
O X .

Player O move: (0, 0) time: 0.0000s
O O .
X . X
O X .

Player X move: (1, 1) time: 0.0000s
O O .
X X X
O X .

CPU times: user 101 ms, sys: 1.62 ms, total: 102 ms
Wall time: 103 ms


O O .
X X X
O X .

In [29]:
%time play_game(TicTacToe(), dict(X=search_player(alphabeta_search), O=search_player(alphabeta_search)), True)

Player X move: (0, 1) time: 0.0949s
. . .
X . .
. . .

Player O move: (2, 1) time: 0.0166s
. . .
X . O
. . .

Player X move: (1, 2) time: 0.0049s
. . .
X . O
. X .

Player O move: (0, 0) time: 0.0015s
O . .
X . O
. X .

Player X move: (1, 1) time: 0.0006s
O . .
X X O
. X .

Player O move: (1, 0) time: 0.0001s
O O .
X X O
. X .

Player X move: (2, 0) time: 0.0001s
O O X
X X O
. X .

Player O move: (0, 2) time: 0.0000s
O O X
X X O
O X .

Player X move: (2, 2) time: 0.0000s
O O X
X X O
O X X

CPU times: user 117 ms, sys: 1.77 ms, total: 119 ms
Wall time: 119 ms


O O X
X X O
O X X

# 3. Heuristic Cutoffs (Optional)

In [30]:
def cutoff_depth(d):
    """A cutoff function that searches to depth d."""
    return lambda game, state, depth: depth > d

# cut depth = 6, meaning we will search 6 moves ahead, and then use the heuristic function to evaluate the state
# if the game tree is too deep, we can use a heuristic function to evaluate the state instead of searching all the way to the leaves
# this is because the game tree grows exponentially with the depth, and we may not have enough time to search all the way to the leaves
# in this way, we can still make a good decision without searching the entire game tree,
# by using a heuristic function to evaluate the state at the cutoff depth.
def h_alphabeta_search(cutoff=cutoff_depth(6), h=lambda s, p: 0):
    def search(game, state):
        """
        Search game to determine best action; use alpha-beta pruning.
        As in [Figure 5.7], this version searches all the way to the leaves.
        TODO: add checks for *cutoff* and return *h* at proper locations
        """

        player = state.to_move

        def max_value(state, alpha, beta, depth):
            if game.is_terminal(state):
                return game.utility(state, player), None
            if cutoff(game, state, depth):
                return h(state, player), None

            v, move = -infinity, None
            for a in game.actions(state):
                v2, _ = min_value(game.result(state, a), alpha, beta, depth+1)

                # TODO: update alpha, beta pruning, decide *v* and *move*

                if v2 > v:
                    v, move = v2, a

                if v >= beta:
                    return v, move

                alpha = max(alpha, v)
                
            return v, move

        def min_value(state, alpha, beta, depth):

            # TODO: implement function min_value

            if game.is_terminal(state):
                return game.utility(state, player), None
            if cutoff(game, state, depth):
                return h(state, player), None

            v, move = +infinity, None

            for a in game.actions(state):
                v2, _ = max_value(game.result(state, a), alpha, beta, depth+1)

                if v2 < v:
                    v, move = v2, a

                if v <= alpha:
                    return v, move
                beta = min(beta, v)
            
            return v, move

        return max_value(state, -infinity, +infinity, 0)
    return search

# plug in your own *h* and see its performance against a random player
game = TicTacToe()

# I'll just cheat with the true utility for a demo (won't work when the tree is deeper)
h = lambda s, p: game.utility(s, p) 
cutoff = cutoff_depth(9)

%time play_game(game, {'X':search_player(h_alphabeta_search(cutoff=cutoff, h=h)), 'O':random_player}, True)

Player X move: (0, 1) time: 0.0955s
. . .
X . .
. . .

Player O move: (0, 2) time: 0.0000s
. . .
X . .
O . .

Player X move: (1, 2) time: 0.0050s
. . .
X . .
O X .

Player O move: (2, 0) time: 0.0000s
. . O
X . .
O X .

Player X move: (1, 1) time: 0.0003s
. . O
X X .
O X .

Player O move: (1, 0) time: 0.0000s
. O O
X X .
O X .

Player X move: (2, 1) time: 0.0000s
. O O
X X X
O X .

CPU times: user 99.8 ms, sys: 1.23 ms, total: 101 ms
Wall time: 101 ms


. O O
X X X
O X .

# 4. Further Hints on Project

* Be **absolutely sure** that you are not maximizing your opponent's utility!
* Code optimization: the performance of a python program greatly depends on its implementation. Write your code to be as efficient as possible and maybe search for performance hacks!